# DefectosCafeVerde grouped — D0DIRECT vs AF2DIRECT — seed 42

Paired validation-only screening from the same official `yolo26n.pt`. The runtime dataset contains only train and validation; test members in the archive are never extracted. Re-running resumes each same-contract arm from Drive `last.pt`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import csv, hashlib, importlib, json, os, shutil, subprocess, sys, tarfile, time
WORK=Path('/content'); REPO=WORK/'coffee-bean-detection'
BRANCH='codex/public-dataset-eligibility-audit'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True,cwd=WORK)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab sebelum Run All')
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
ARCHIVE_NAME='defectoscafeverde-grouped-physical-v1.tar'
EXPECTED_ARCHIVE_SHA='53fb2233f1f0d1c77cb24eca2d720f86e0a16835b8a69f4e8f3176fae1aacef2'
matches=sorted(Path('/content/drive/MyDrive').rglob(ARCHIVE_NAME))
if len(matches)!=1: raise FileNotFoundError(f'Harus tepat satu {ARCHIVE_NAME} di MyDrive; ditemukan {matches}')
ARCHIVE=matches[0]
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
if sha256(ARCHIVE)!=EXPECTED_ARCHIVE_SHA: raise RuntimeError('SHA archive DefectosCafeVerde tidak cocok')
print('ARCHIVE VALID:',ARCHIVE)


In [ ]:
DATA=WORK/'defectoscafeverde-development-v1'
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir(parents=True)
prefix='grouped-physical-v1/'
allowed_files={'data.yaml','grouped_audit.json','grouped_summary.json'}
with tarfile.open(ARCHIVE,'r') as bundle:
    for member in bundle.getmembers():
        name=member.name.replace('\\','/')
        if not name.startswith(prefix): continue
        relative=name[len(prefix):].strip('/')
        if not relative: continue
        first=relative.split('/',1)[0]
        allowed=first in {'train','val'} or relative in allowed_files
        if not allowed: continue
        target=(DATA/relative).resolve()
        if DATA.resolve() not in target.parents and target!=DATA.resolve(): raise RuntimeError('Unsafe archive path')
        if member.isdir(): target.mkdir(parents=True,exist_ok=True); continue
        target.parent.mkdir(parents=True,exist_ok=True)
        source=bundle.extractfile(member)
        if source is None: raise RuntimeError(f'Gagal membaca {member.name}')
        with source, target.open('wb') as out: shutil.copyfileobj(source,out)
if (DATA/'test').exists(): raise RuntimeError('TEST TEREXPOSE — STOP')
import yaml
payload=yaml.safe_load((DATA/'data.yaml').read_text())
payload.pop('test',None); payload['path']=str(DATA); payload['train']='train/images'; payload['val']='val/images'
(DATA/'data.yaml').write_text(yaml.safe_dump(payload,sort_keys=False),encoding='utf-8')
AUDIT=DATA/'grouped_audit.json'
print('TRAIN:',len(list((DATA/'train/images').glob('*'))),'VAL:',len(list((DATA/'val/images').glob('*'))))
print('TEST PRESENT:',(DATA/'test').exists(),'YAML TEST KEY:','test' in payload)


In [ ]:
from ultralytics import YOLO
_=YOLO('yolo26n.pt')
PRETRAINED=(REPO/'yolo26n.pt').resolve()
print('PRETRAINED:',PRETRAINED,'SHA:',sha256(PRETRAINED))
OUT=Path('/content/drive/MyDrive/Coffee_Bean_Detection/experiments/defectoscafeverde-af2-direct-v1')
OUT.mkdir(parents=True,exist_ok=True)
from coffee_detector.experiments.run_defectoscafeverde_af2_direct import run_static_preflight, validate_development_dataset
contract=validate_development_dataset(DATA,AUDIT)
static=run_static_preflight(PRETRAINED,OUT/'static_preflight_notebook.json',seed=42)
print('DATA GATES:',contract['gates'])
print('STATIC GATES:',static['gates'])
print('STATIC DECISION:',static['decision'])


In [ ]:
LOG=OUT/'paired_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_defectoscafeverde_af2_direct','--data-root',str(DATA),'--grouped-audit',str(AUDIT),'--pretrained-checkpoint',str(PRETRAINED),'--output-root',str(OUT),'--seed','42','--device','0','--authorize-training']
print('START/RESUME PAIRED SCREEN | log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream:
    process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
def epochs(arm):
    path=OUT/arm/f'{arm}_seed42'/'results.csv'
    if not path.is_file(): return 0
    try:
        with path.open(newline='',encoding='utf-8') as f: return sum(1 for _ in csv.DictReader(f))
    except Exception: return 0
shown=None
while process.poll() is None:
    status=(epochs('D0DIRECT'),epochs('AF2DIRECT'))
    if status!=shown:
        print(f'D0DIRECT {status[0]}/50 | AF2DIRECT {status[1]}/50',flush=True); shown=status
    time.sleep(60)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-150:])); raise RuntimeError(f'Training gagal rc={process.returncode}')
print('PAIRED SCREEN SELESAI')


In [ ]:
from IPython.display import display
import pandas as pd
SUMMARY=OUT/'defectoscafeverde_af2_direct_seed42_summary.json'
result=json.loads(SUMMARY.read_text())
rows=[]
for model,metrics in result['values'].items(): rows.append({'model':model,**metrics})
display(pd.DataFrame(rows).style.format({'macro_map50_95':'{:.2%}','bottom3_class_map50_95':'{:.2%}','worst_class_map50_95':'{:.2%}'}))
print('DELTAS:',result['deltas'])
print('SCREEN:',result['screen'])
print('TEST OPENED:',result['test_opened'])
print('Kirim tabel, deltas, dan SCREEN. Jangan membuka test.')
